In [63]:
def pp(msg=None, value=None):
    if msg is not None:
        print(msg)
    if value is not None:
        print(value)
    print('=' * 100)

The four possible verdicts are: Holds, Holds with reservations, Does not hold, and Cannot be
concluded from this table.

Claim 1: "A night in Stockholm costs approximately 1,791."

Verdict: Holds with reservations
- 1791.00 is a correct median price, but only for 65% of listings (35% don't have any price, because mostly of them are unavailable)

In [64]:
import pandas as pd

listings = pd.read_csv('listings_clean.csv', parse_dates=['first_review', 'last_review'])
pp(listings.shape)
# pp(listings.dtypes)

prices = listings['price']
pp('Statistics', prices.describe())
pp('% of missing prices', prices.isna().mean() * 100)
pp('Availability for listings without prices', 
   listings[prices.isna()]['availability_365'].describe())


(4574, 73)
Statistics
count     2957.000000
mean      2327.643382
std       2160.353619
min        120.000000
25%       1071.120000
50%       1791.000000
75%       3003.800000
max      40000.000000
Name: price, dtype: float64
% of missing prices
35.35198950590293
Availability for listings without prices
count    1617.000000
mean       12.543599
std        61.282241
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       365.000000
Name: availability_365, dtype: float64


Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."

Verdict: Does not hold
- most of 5* ratings (~91%) have less than 20 reviews (~58% of others have 20 or more)

In [65]:
pp(listings['review_scores_rating'].describe())
pp(listings['has_reviews'].value_counts())
perfect = listings[listings['review_scores_rating'] == 5.00]
others = listings[listings['review_scores_rating'] < 5.00]
pp('5*', perfect['number_of_reviews'].describe())
pp('1-4**', others['number_of_reviews'].describe())
pp((perfect['number_of_reviews'] >= 20).mean() * 100)
pp((others['number_of_reviews'] >= 20).mean() * 100)


count    3930.000000
mean        4.817692
std         0.317372
min         1.000000
25%         4.750000
50%         4.920000
75%         5.000000
max         5.000000
Name: review_scores_rating, dtype: float64
has_reviews
True     3930
False     644
Name: count, dtype: int64
5*
count    1484.000000
mean        7.704178
std        10.268451
min         1.000000
25%         2.000000
50%         4.000000
75%         9.000000
max       159.000000
Name: number_of_reviews, dtype: float64
1-4**
count    2446.000000
mean       62.185609
std        99.560995
min         1.000000
25%        11.000000
50%        25.000000
75%        65.000000
max      1293.000000
Name: number_of_reviews, dtype: float64
8.76010781671159
57.72690106295993


Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in
2024."

Verdict: Cannot be concluded from this table
- the data cannot show actual market growth as it only includes present listings (most of lisnings from 2016 do not have any reviews last years)

In [66]:
pp(listings['first_review'].dt.year.value_counts().sort_index())
pp('Listing with first_review in 2016', listings.loc[
        listings['first_review'].dt.year == 2016,
        'last_review'
    ].dt.year.value_counts().sort_index()
)

first_review
2011.0      2
2012.0     12
2013.0     26
2014.0     51
2015.0    116
2016.0    131
2017.0    148
2018.0    191
2019.0    234
2020.0     96
2021.0    159
2022.0    332
2023.0    565
2024.0    610
2025.0    542
2026.0    715
Name: count, dtype: int64
Listing with first_review in 2016
last_review
2016     4
2017     5
2018     5
2019     7
2020     1
2021     5
2022     5
2023     7
2024     8
2025    14
2026    70
Name: count, dtype: int64


Claim 4: "Most hosts in Stockholm are professional companies with many listings."

Verdict: Does not hold
- ~90.14% of hosts have only one listing
- only ~9.86% have two or more listings
- dataset does not contain a column identifying professional companies

In [67]:
host_counts = listings.groupby("host_id").size()
pp("Total hosts", host_counts.shape[0])

one_listing = (host_counts == 1).sum()
pp("Hosts with 1 listing", one_listing)

multiple_listings = (host_counts >= 2).sum()
pp("Hosts with 2 or more listings", multiple_listings)

pp("Difference (%)", one_listing * 100 / len(host_counts))

prof_columns = [col for col in listings.columns
                if "professional" in col.lower()]
pp("Columns, describing a professional host", prof_columns) # []

Total hosts
3591
Hosts with 1 listing
3237
Hosts with 2 or more listings
354
Difference (%)
90.14202172096908
Columns, describing a professional host
[]


Task 5 is different: a data note, not a claim
There is nothing to test here, and no verdict. Instead you write about the whole table,
using what you found in claims 1 to 4. Write a short text in a markdown cell, at most ten
sentences, as if it was the first paragraph of a report about this table. Answer in your own
words: what is missing (with a number)? Who or what is not in the table? Who made the
data, when, and under which licence? Which columns are calculated and not measured?
And finally: what can this table not tell us? Finish with one sentence: which of the four
claims would you most want to correct if you read it in a newspaper, and why?

In [68]:
estimated_columns = [col for col in listings.columns 
                     if col.lower().startswith("estimated_")]
pp("Estimated columns", estimated_columns)

calculated_columns = [col for col in listings.columns 
                      if col.lower().startswith("calculated_")]
pp("Calculated columns", calculated_columns)

Estimated columns
['estimated_occupancy_l365d', 'estimated_revenue_l365d']
Calculated columns
['calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms']


The table contains 4,574 Airbnb listings in Stockholm, but some information is missing, including the price for 1,617 listings (~35%). The listings in this file were scraped on 26 and 27 September 2026. Some columns contain estimates or calculations rather than directly measured information, including estimated_occupancy_l365d, estimated_revenue_l365d and the four columns beginning with calculated_. These figures should therefore not be treated as exact measurements of actual bookings or income. The table cannot tell us exactly how many nights properties were actually booked or whether hosts are professional companies. It also cannot explain why hosts rent out their properties or how Airbnb affects the housing market. Of the four claims, I would most want to correct claim 3 in a newspaper because we cannot decide about if market has grown due to lacking of data. Second will be claim 4 as the data does not provide any information if a host is professional or not.